# Comparaison des modèles (Addiction Population Data)

**Dataset :** Addiction Population Data  
**Variable cible :** `smokes_per_day` nombre de cigarettes consommés par jours 

Ce notebook charge les données, fait le preprocessing, puis appelle chaque modèle depuis le dossier `models/`.  
Il compare ensuite les résultats et propose un exemple d'application concret.

## 1 - Importation des bibliothèques

In [ ]:
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from sklearn.preprocessing import StandardScaler

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                             r2_score)

from sklearn.linear_model import SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import VotingClassifier
from sklearn import tree

import sys
sys.path.append('..')

from sklearn.model_selection import train_test_split
from models.encoder import labelEncoder, oneHotEncoder
from models.ACP import ACP

## 2 - Chargement et preprocessing

On charge le dataset et on convertit les variables textuelles en nombres avec `LabelEncoder`.  
Ensuite on sépare les données en 80% entraînement / 20% test.

In [ ]:
df = pd.read_csv('BDD_initial/addiction_population_data.csv')

X_raw = df.drop(columns=['smokes_per_day'])
y     = df['smokes_per_day']

# LabelEncoder
X_le  = labelEncoder(X_raw.copy())

# OneHotEncoder
X_ohe = oneHotEncoder(X_raw.copy())

# ACP
x_acp = ACP(X_ohe, n_components=50)

# Split — même random_state pour comparer sur les mêmes données
X_train_le,  X_test_le,  y_train, y_test = train_test_split(X_le,  y, test_size=0.2, random_state=42)
X_train_ohe, X_test_ohe, _,       _      = train_test_split(X_ohe, y, test_size=0.2, random_state=42)

#Split - ACP
X_train_acp, X_test_acp, _, _ = train_test_split(x_acp, y, test_size=0.2, random_state=42)

# label_encoders conservé pour l'exemple étudiant (cell 16)
from sklearn.preprocessing import LabelEncoder
cat_cols = df.select_dtypes(include='object').columns.tolist()
label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    le.fit(df[col])
    label_encoders[col] = le

print('X_le  shape:', X_le.shape)
print('X_ohe shape:', X_ohe.shape)
print('X_acp shape:', x_acp.shape)
print('Train / Test:', X_train_le.shape[0], '/', X_test_le.shape[0])

X_le  shape: (3000, 26)
X_ohe shape: (3000, 5913)
X_acp shape: (3000, 50)
Train / Test: 2400 / 600


C:\Users\lanat\AppData\Local\Temp\ipykernel_2608\4256162178.py:24: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include='object').columns.tolist()


## 3 - Qu'est-ce que le Random Forest ?

### Principe général

Le Random Forest est un algorithme d'apprentissage par ensemble (ensemble learning).  
Plutôt que de se fier à un seul arbre de décision, on en construit un grand nombre, chacun entraîné dans des conditions légèrement différentes, puis on combine leurs prédictions.

### Fonctionnement détaillé

**1. Bootstrap sampling**  
Pour construire chaque arbre, on tire aléatoirement un sous-ensemble des données d'entraînement *avec remise* — certains exemples apparaissent plusieurs fois, d'autres pas du tout. Chaque arbre voit donc une version légèrement différente du dataset.

**2. Sélection aléatoire des variables**  
À chaque nœud de chaque arbre, l'algorithme ne considère pas toutes les variables disponibles, mais seulement un sous-ensemble aléatoire (par défaut : √n_features). Cela force les arbres à être différents les uns des autres.

**3. Agrégation (bagging)**  
Pour une prédiction finale, chaque arbre donne sa propre estimation du score. Le Random Forest retourne la **moyenne** de toutes ces prédictions — ce qui réduit la variance et rend le modèle plus robuste qu'un arbre unique.

### Pourquoi c'est efficace ?

Un arbre de décision seul a tendance à **surapprendere** (overfitting) : il mémorise les données d'entraînement au lieu d'apprendre une structure générale.  
En combinant des centaines d'arbres entraînés différemment, le Random Forest **moyenne les erreurs individuelles** — les erreurs de certains arbres sont compensées par les autres.

### Hyperparamètres principaux

| Paramètre | Rôle |
|---|---|
| `n_estimators` | Nombre d'arbres dans la forêt |
| `max_depth` | Profondeur maximale de chaque arbre |
| `min_samples_split` | Nombre minimum d'exemples pour diviser un nœud |
| `min_samples_leaf` | Nombre minimum d'exemples dans une feuille |

## 4. Entraînement des modèles

Chaque modèle est dans un fichier séparé dans `models/`.  
La fonction `run(X_train, X_test, y_train, y_test)` retourne un dictionnaire avec les métriques.

> **Pour ajouter un modèle** : importer sa fonction `run` et l'ajouter au dictionnaire `modeles`.

In [ ]:
from models.random_forest import run as run_rf
from models.xgboost import run as run_xgb
from models.knn_regressor import run as run_knn
from models.naive_bayes_regressor import run as run_nb

modeles = {
    'KNN'          : run_knn,
    'XGBoost'      : run_xgb,
    'Naive Bayes'  : run_nb,
    'Random Forest': run_rf,
}

resultats = {}
for nom, run_fn in modeles.items():
    print(f'Entraînement : {nom}...')
    resultats[f'{nom} (LE)']  = run_fn(X_train_le,  X_test_le,  y_train, y_test)
    resultats[f'{nom} (ACP)'] = run_fn(X_train_acp, X_test_acp, y_train, y_test)
    print(f"  LE  R²: {resultats[f'{nom} (LE)']['optimized']['r2']:.4f}")
    print(f"  ACP R²: {resultats[f'{nom} (ACP)']['optimized']['r2']:.4f}")

print('\nTerminé.')


Entraînement : KNN...
  LE  R²: 0.4884
  ACP R²: -0.0397
Entraînement : XGBoost...


## 5. Tableau comparatif

On compare les modèles sur trois métriques :
- **R²** : proportion de variance expliquée (plus c'est proche de 1, mieux c'est)
- **MAE** : erreur moyenne en points de score
- **RMSE** : similaire au MAE, mais pénalise plus les grandes erreurs

In [ ]:
rows = []
for nom, res in resultats.items():
    opt = res['optimized']
    rows.append({
        'Modèle': nom,
        'R²'    : round(opt['r2'],   4),
        'MAE'   : round(opt['mae'],  4),
        'RMSE'  : round(opt['rmse'], 4),
    })

df_comparaison = pd.DataFrame(rows).sort_values('R²', ascending=False).set_index('Modèle')
df_comparaison

## 6. Visualisation des métriques

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

metrics = [('R²', '#2ecc71', False), ('MAE', '#e74c3c', True), ('RMSE', '#e67e22', True)]

for ax, (metric, color, ascending) in zip(axes, metrics):
    vals = df_comparaison[metric].sort_values(ascending=ascending)
    ax.barh(vals.index, vals.values, color=color, alpha=0.8)
    ax.set_title(metric, fontsize=13)
    ax.set_xlabel('Valeur')

plt.suptitle('Comparaison des modèles — Addicted_Score', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 7. Analyse détaillée : Random Forest

### Importance des variables

Le Random Forest calcule automatiquement l'importance de chaque variable :  
plus le score est élevé, plus la variable influence la prédiction.

In [ ]:
# Feature importances — uniquement pour les modèles qui les fournissent
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

modeles_avec_importances = {
    k: v['feature_importances']
    for k, v in resultats.items()
    if v.get('feature_importances') is not None and '(LE)' in k
}

for ax, (nom, importances) in zip(axes, modeles_avec_importances.items()):
    importances.sort_values().plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(f'Importance des variables — {nom}', fontsize=11)
    ax.set_xlabel('Importance')

plt.tight_layout()
plt.show()


### Valeurs réelles vs valeurs prédites

Un modèle parfait alignerait tous les points sur la diagonale rouge.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

for i, encoder in enumerate(['LE', 'OHE']):
    key    = f'Random Forest ({encoder})'
    model  = resultats[key]['model']
    X_test_enc = X_test_le if encoder == 'LE' else X_test_ohe
    y_pred = model.predict(X_test_enc)
    r2     = resultats[key]['optimized']['r2']
    residus = y_test - y_pred

    axes[i,0].scatter(y_test, y_pred, alpha=0.5, color='steelblue', edgecolors='k', linewidths=0.3)
    axes[i,0].plot([y.min(), y.max()], [y.min(), y.max()], 'r--', linewidth=1.5, label='Prédiction parfaite')
    axes[i,0].set_xlabel('Valeurs réelles')
    axes[i,0].set_ylabel('Valeurs prédites')
    axes[i,0].set_title(f'Réel vs Prédit — {encoder}  (R² = {r2:.3f})')
    axes[i,0].legend()

    axes[i,1].hist(residus, bins=20, color='steelblue', edgecolor='white')
    axes[i,1].axvline(0, color='red', linestyle='--', linewidth=1.5)
    axes[i,1].set_xlabel('Résidu (réel − prédit)')
    axes[i,1].set_ylabel('Fréquence')
    axes[i,1].set_title(f'Résidus — {encoder}')
    print(f'{encoder}  →  Résidu moyen: {residus.mean():.4f}   Écart-type: {residus.std():.4f}')

plt.tight_layout()
plt.show()

## 8. Exemple d'application

On crée un profil d'étudiant fictif et on demande au modèle de prédire son score d'addiction.  
Les variables catégorielles sont encodées avec les mêmes `LabelEncoder` utilisés à l'étape 2.

In [ ]:
# exemple d'un étudiant fictif
etudiant = {
    'Age'                          : 20,
    'Gender'                       : 'Male',
    'Academic_Level'               : 'Undergraduate',
    'Country'                      : 'USA',
    'Avg_Daily_Usage_Hours'        : 5.0,
    'Most_Used_Platform'           : 'Instagram',
    'Affects_Academic_Performance' : 'Yes',
    'Sleep_Hours_Per_Night'        : 6.0,
    'Mental_Health_Score'          : 5,
    'Relationship_Status'          : 'Single',
    'Conflicts_Over_Social_Media'  : 2,
}

# trouve le meilleur modèle selon R²
meilleur_cle = max(resultats, key=lambda k: resultats[k]['optimized']['r2'])
meilleur_r2  = resultats[meilleur_cle]['optimized']['r2']

etudiant_df = pd.DataFrame([etudiant])

# encodage
etudiant_enc = {}
for col, val in etudiant.items():
    if col in label_encoders:
        etudiant_enc[col] = label_encoders[col].transform([val])[0]
    else:
        etudiant_enc[col] = val

if '(LE)' in meilleur_cle:
    X_etudiant = pd.DataFrame([etudiant_enc])[X_le.columns]
else:
    etudiant_ohe_raw = oneHotEncoder(etudiant_df.copy())
    X_etudiant       = etudiant_ohe_raw.reindex(columns=X_ohe.columns, fill_value=0)

score_predit = resultats[meilleur_cle]['model'].predict(X_etudiant)[0]

print("=== Profil de l'étudiant ===")
for k, v in etudiant.items():
    print(f"  {k:<35} : {v}")

print(f"\n Modèle utilisé : {meilleur_cle}  (R² = {meilleur_r2:.4f})")
print(f" Score prédit   : {score_predit:.2f} / 9")

if score_predit >= 7:
    print("    → Niveau élevé d'addiction")
elif score_predit >= 4:
    print("    → Niveau modéré d'addiction")
else:
    print("    → Niveau faible d'addiction")

## 9. Conclusion

Ce notebook compare les modèles de régression entraînés sur le dataset `Students Social Media Addiction`.  
Le meilleur modèle est sélectionné selon le R² le plus élevé et le MAE/RMSE les plus faibles.

**Pour intégrer un nouveau modèle :**
1. Créer `models/nom_du_modele.py` avec une fonction `run(X_train, X_test, y_train, y_test) -> dict`
2. La fonction doit retourner au minimum : `{'optimized': {'r2', 'mae', 'rmse'}, 'model': estimateur}`
3. Décommenter la ligne d'import correspondante dans la cellule 4